# Exercício 7 — Recursão para navegação em árvore de diretórios

Aluno: Henrique Goldstein Maluhy Mendes Amaral · DR1_AT · Infnet

Para rodar: Ambiente de execução → Executar tudo.

Tudo é feito em memória: o notebook não lê nem escreve nenhum arquivo de verdade.

In [1]:
import random
import sys
import time

import pandas as pd

random.seed(42)

## 1. A estrutura

Cada nó tem nome, tipo (`"dir"` ou `"arquivo"`) e filhos. Só diretório tem filhos.

In [2]:
class NoArquivo:
    def __init__(self, nome, tipo):
        self.nome = nome
        self.tipo = tipo
        self.filhos = []

    def eh_diretorio(self):
        return self.tipo == "dir"

    def filho(self, nome):
        """Procura um filho pelo nome (busca linear entre os filhos)."""
        for f in self.filhos:
            if f.nome == nome:
                return f
        return None

## 2. O caminho atual numa lista encadeada

Durante a travessia, ao entrar num diretório empilho o nome dele e ao sair desempilho. A pilha é uma lista encadeada: push e pop mexem só no topo, O(1), e montar o caminho completo percorre a pilha, O(d), onde d é a profundidade.

In [3]:
class NoCaminho:
    def __init__(self, nome, abaixo):
        self.nome = nome
        self.abaixo = abaixo


class PathStack:
    """Pilha de nomes em lista encadeada: o topo é a pasta mais funda."""
    def __init__(self):
        self.topo = None
        self.tamanho = 0

    def push(self, nome):
        self.topo = NoCaminho(nome, self.topo)
        self.tamanho += 1

    def pop(self):
        if self.topo is None:
            raise IndexError("pop em pilha de caminho vazia")
        nome = self.topo.nome
        self.topo = self.topo.abaixo
        self.tamanho -= 1
        return nome

    def caminho(self):
        # o topo é o nome mais fundo, então junto de cima para baixo e inverto
        nomes = []
        atual = self.topo
        while atual is not None:
            nomes.append(atual.nome)
            atual = atual.abaixo
        return "/" + "/".join(reversed(nomes))

## 3. `walk(root)` recursivo

Pré-ordem: registra o nó e depois desce em cada filho. O caso base é um nó sem filhos, em que o laço não roda e a recursão para.

In [4]:
def walk(root):
    caminhos = []
    pilha = PathStack()

    def visitar(no):
        if no is not root:
            pilha.push(no.nome)                 # entrei neste nó
        caminhos.append(pilha.caminho())
        for filho in no.filhos:                 # caso base: sem filhos, nada a fazer
            visitar(filho)
        if no is not root:
            pilha.pop()                         # saí dele: volto para o pai

    visitar(root)
    return caminhos

## 4. `DirectoryTree`

Validações: o caminho tem que ser absoluto, sem partes vazias e sem `.` ou `..`; no insert, o pai tem que existir e ser diretório, e o nome ainda não pode existir; o tipo só pode ser `"dir"` ou `"arquivo"`; a raiz não pode ser inserida nem removida.

Política de remoção: remover um diretório com conteúdo exige `recursive=True`, senão levanta `DiretorioNaoVazioError` (como o `rmdir` e o `rm -r`). Apagar sem querer uma pasta cheia perde tudo que está dentro, então isso precisa ser pedido de forma explícita. Arquivo e diretório vazio são removidos direto.

In [5]:
class DiretorioNaoVazioError(Exception):
    """Tentativa de remover um diretório com conteúdo sem recursive=True."""


class DirectoryTree:
    def __init__(self):
        self.root = NoArquivo("", "dir")

    def _partes(self, path):
        if not isinstance(path, str) or not path.startswith("/"):
            raise ValueError(f"o caminho precisa ser absoluto (começar com '/'): {path!r}")
        if path == "/":
            return []
        partes = path.strip("/").split("/")
        for parte in partes:
            if parte in ("", ".", ".."):
                raise ValueError(f"parte inválida {parte!r} no caminho {path!r}")
        return partes

    def search(self, path):
        atual = self.root
        for nome in self._partes(path):
            if not atual.eh_diretorio():
                return None                     # tentei entrar dentro de um arquivo
            atual = atual.filho(nome)
            if atual is None:
                return None
        return atual

    def insert(self, path, node_type):
        if node_type not in ("dir", "arquivo"):
            raise ValueError(f"tipo inválido {node_type!r}: use 'dir' ou 'arquivo'")
        partes = self._partes(path)
        if not partes:
            raise ValueError("a raiz já existe e não pode ser inserida")
        pai_path = "/" + "/".join(partes[:-1])
        pai = self.search(pai_path)
        if pai is None:
            raise FileNotFoundError(f"o diretório pai {pai_path!r} não existe")
        if not pai.eh_diretorio():
            raise NotADirectoryError(f"{pai_path!r} é um arquivo, não pode ter filhos")
        if pai.filho(partes[-1]) is not None:
            raise FileExistsError(f"{path!r} já existe")
        novo = NoArquivo(partes[-1], node_type)
        pai.filhos.append(novo)
        return novo

    def delete(self, path, recursive=False):
        """Remove e devolve quantos nós saíram da árvore."""
        partes = self._partes(path)
        if not partes:
            raise ValueError("a raiz não pode ser removida")
        pai = self.search("/" + "/".join(partes[:-1]))
        alvo = pai.filho(partes[-1]) if pai is not None and pai.eh_diretorio() else None
        if alvo is None:
            raise FileNotFoundError(f"{path!r} não existe")
        if alvo.filhos and not recursive:
            raise DiretorioNaoVazioError(
                f"{path!r} tem {len(alvo.filhos)} item(ns); use recursive=True para apagar tudo")
        removidos = contar(alvo)
        pai.filhos.remove(alvo)
        return removidos


def contar(no):
    """Quantos nós há na subárvore (recursivo, mesma forma do walk)."""
    return 1 + sum(contar(filho) for filho in no.filhos)

### Montando uma árvore de exemplo

In [6]:
arvore = DirectoryTree()
for path, tipo in [
    ("/projeto", "dir"),
    ("/projeto/src", "dir"),
    ("/projeto/src/main.py", "arquivo"),
    ("/projeto/src/util.py", "arquivo"),
    ("/projeto/tests", "dir"),
    ("/projeto/tests/test_main.py", "arquivo"),
    ("/projeto/README.md", "arquivo"),
    ("/docs", "dir"),
    ("/docs/manual.pdf", "arquivo"),
    ("/vazia", "dir"),
]:
    arvore.insert(path, tipo)

for caminho in walk(arvore.root):
    no = arvore.search(caminho)
    print(f"{caminho:<30} {'dir' if no.eh_diretorio() else 'arquivo'}")

/                              dir
/projeto                       dir
/projeto/src                   dir
/projeto/src/main.py           arquivo
/projeto/src/util.py           arquivo
/projeto/tests                 dir
/projeto/tests/test_main.py    arquivo
/projeto/README.md             arquivo
/docs                          dir
/docs/manual.pdf               arquivo
/vazia                         dir


In [7]:
esperado = ["/", "/projeto", "/projeto/src", "/projeto/src/main.py", "/projeto/src/util.py",
            "/projeto/tests", "/projeto/tests/test_main.py", "/projeto/README.md",
            "/docs", "/docs/manual.pdf", "/vazia"]
assert walk(arvore.root) == esperado          # pré-ordem, na ordem de inserção

# busca
assert arvore.search("/projeto/src/util.py").nome == "util.py"
assert arvore.search("/projeto/nada") is None
assert arvore.search("/projeto/README.md/dentro") is None   # não existe nada "dentro" de um arquivo

# validações do insert
for path, tipo, erro_esperado in [
    ("/projeto/src", "dir", FileExistsError),
    ("/inexistente/a.txt", "arquivo", FileNotFoundError),
    ("/projeto/README.md/x", "arquivo", NotADirectoryError),
    ("projeto/b", "dir", ValueError),
    ("/projeto//b", "dir", ValueError),
    ("/projeto/../b", "dir", ValueError),
    ("/projeto/b", "pasta", ValueError),
]:
    try:
        arvore.insert(path, tipo)
        raise AssertionError(f"insert({path!r}) deveria falhar")
    except erro_esperado as erro:
        print(f"insert({path!r}) → {type(erro).__name__}: {erro}")

insert('/projeto/src') → FileExistsError: '/projeto/src' já existe
insert('/inexistente/a.txt') → FileNotFoundError: o diretório pai '/inexistente' não existe
insert('/projeto/README.md/x') → NotADirectoryError: '/projeto/README.md' é um arquivo, não pode ter filhos
insert('projeto/b') → ValueError: o caminho precisa ser absoluto (começar com '/'): 'projeto/b'
insert('/projeto//b') → ValueError: parte inválida '' no caminho '/projeto//b'
insert('/projeto/../b') → ValueError: parte inválida '..' no caminho '/projeto/../b'
insert('/projeto/b') → ValueError: tipo inválido 'pasta': use 'dir' ou 'arquivo'


### A política de remoção em ação

In [8]:
# diretório com conteúdo, sem a flag: recusa e não mexe em nada
antes = walk(arvore.root)
try:
    arvore.delete("/projeto/src")
except DiretorioNaoVazioError as erro:
    print(f"delete('/projeto/src') → {erro}")
assert walk(arvore.root) == antes

# com recursive=True, a subárvore inteira sai
print(f"delete('/projeto/src', recursive=True) removeu {arvore.delete('/projeto/src', recursive=True)} nós")
assert arvore.search("/projeto/src") is None and arvore.search("/projeto/src/main.py") is None

# arquivo e diretório vazio saem direto
assert arvore.delete("/docs/manual.pdf") == 1
assert arvore.delete("/vazia") == 1

# o resto continua consistente
print("\ndepois das remoções:")
for caminho in walk(arvore.root):
    print(" ", caminho)
assert walk(arvore.root) == ["/", "/projeto", "/projeto/tests", "/projeto/tests/test_main.py",
                             "/projeto/README.md", "/docs"]

delete('/projeto/src') → '/projeto/src' tem 2 item(ns); use recursive=True para apagar tudo
delete('/projeto/src', recursive=True) removeu 3 nós

depois das remoções:
  /
  /projeto
  /projeto/tests
  /projeto/tests/test_main.py
  /projeto/README.md
  /docs


## 5. Pilha de caminho: lista encadeada × lista do Python

Nas operações fundamentais: push e pop são O(1) nas duas; montar o caminho é O(d) nas duas, mas a lista encadeada precisa percorrer os nós e inverter a ordem; acessar o nível i é O(1) na lista do Python e O(d) na encadeada. Rodo a mesma travessia com as duas:

In [9]:
def gerar_arvore(n):
    """Árvore aleatória com n nós: cada novo item vai para um diretório sorteado."""
    t = DirectoryTree()
    diretorios = [""]                  # "" representa a raiz
    for i in range(n):
        pai = random.choice(diretorios)
        eh_dir = random.random() < 0.4
        caminho = f"{pai}/{'d' if eh_dir else 'f'}{i}"
        t.insert(caminho, "dir" if eh_dir else "arquivo")
        if eh_dir:
            diretorios.append(caminho)
    return t


def walk_com_lista_python(root):
    """Mesmo walk, mas com a pilha do caminho numa lista do Python."""
    caminhos, pilha = [], []

    def visitar(no):
        if no is not root:
            pilha.append(no.nome)
        caminhos.append("/" + "/".join(pilha))
        for filho in no.filhos:
            visitar(filho)
        if no is not root:
            pilha.pop()

    visitar(root)
    return caminhos


linhas = []
for n in [1_000, 5_000, 20_000]:
    t = gerar_arvore(n)

    inicio = time.perf_counter()
    com_encadeada = walk(t.root)
    tempo_encadeada = time.perf_counter() - inicio

    inicio = time.perf_counter()
    com_python = walk_com_lista_python(t.root)
    tempo_python = time.perf_counter() - inicio

    assert com_encadeada == com_python            # mesmo resultado, caminho por caminho
    profundidades = [c.count("/") for c in com_encadeada[1:]]
    linhas.append({
        "nos": n + 1,
        "push e pop (cada)": n,
        "profundidade_media": sum(profundidades) / n,
        "passos_para_montar_caminhos": sum(profundidades),
        "tempo_encadeada_ms": tempo_encadeada * 1000,
        "tempo_lista_python_ms": tempo_python * 1000,
        "encadeada / python": tempo_encadeada / tempo_python,
    })

pd.DataFrame(linhas)

,nos,push e pop (cada),profundidade_media,passos_para_montar_caminhos,tempo_encadeada_ms,tempo_lista_python_ms,encadeada / python
0,1001,1000,6.95100,6951,0.8020,0.2533,3.166206
1,5001,5000,7.63440,38172,4.1344,1.1185,3.696379
2,20001,20000,8.76485,175297,45.9861,14.8905,3.088284


As duas fazem as mesmas operações (um push e um pop por nó) e devolvem os mesmos caminhos. O walk é O(n·d), porque monta um caminho de tamanho d para cada nó. A lista encadeada foi mais lenta no tempo, porque para montar o caminho precisa percorrer os nós e inverter a ordem, enquanto a lista do Python já guarda os nomes na ordem certa. Para uma pilha de caminho, a lista do Python é a escolha prática.

## 6. O limite da recursão

O walk usa uma chamada por nível. Com mais níveis que o limite de recursão do Python, ele quebra:

In [10]:
limite = sys.getrecursionlimit()
funda = DirectoryTree()
caminho = ""
for i in range(limite + 500):
    caminho += f"/n{i}"
    funda.insert(caminho, "dir")
print(f"limite de recursão: {limite}; árvore com {limite + 500} níveis")
try:
    walk(funda.root)
    print("walk: terminou")
except RecursionError:
    print("walk recursivo: RecursionError — a árvore é mais funda que o limite")

limite de recursão: 1000; árvore com 1500 níveis
walk recursivo: RecursionError — a árvore é mais funda que o limite


Na prática a profundidade de diretórios raramente passa de algumas dezenas, então a recursão é segura. Para árvores muito fundas, uma versão iterativa com pilha explícita não quebra.

Custos (n nós, d profundidade, k filhos por diretório):
- walk: O(n·d)
- search e insert: O(d·k)
- delete: O(d·k) para achar, mais o tamanho da subárvore removida
- push e pop do caminho: O(1)

## 7. Casos de borda

In [11]:
t = DirectoryTree()
assert walk(t.root) == ["/"]                  # árvore só com a raiz
assert t.search("/") is t.root

for chamada, erro_esperado in [
    (lambda: t.delete("/"), ValueError),               # a raiz não se remove
    (lambda: t.insert("/", "dir"), ValueError),        # nem se insere de novo
    (lambda: t.delete("/nada"), FileNotFoundError),    # remover o que não existe
    (lambda: t.search(None), ValueError),              # caminho que nem é texto
]:
    try:
        chamada()
        raise AssertionError("deveria ter falhado")
    except erro_esperado:
        pass

# barra no final é aceita
t.insert("/a/", "dir")
assert t.search("/a") is not None and t.search("/a/") is not None

# nomes iguais em diretórios diferentes não conflitam
t.insert("/b", "dir")
t.insert("/a/x", "arquivo")
t.insert("/b/x", "arquivo")
assert walk(t.root) == ["/", "/a", "/a/x", "/b", "/b/x"]

# pilha de caminho vazia
try:
    PathStack().pop()
except IndexError as erro:
    print(f"pop na pilha vazia → {erro}")
assert PathStack().caminho() == "/"

print("todos os casos de borda passaram")

pop na pilha vazia → pop em pilha de caminho vazia
todos os casos de borda passaram
